In [1]:
import torch

# preprocessor
processor = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_preprocessor')
# models
vjepa2_vit_large = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_large')
# vjepa2_vit_huge = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_huge')
# vjepa2_vit_giant = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant')
# vjepa2_vit_giant_384 = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')



Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main
Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main
/home/wjl/.conda/envs/jepa_torch/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
import json

path = "data/ssv2/labels/train.json"

with open(path, "r") as f:
    data = json.load(f)

print(data)



In [ ]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader



from app.vjepa.utils import init_opt, init_video_model, load_checkpoint

encoder, predictor = init_video_model(
    device="cpu",
    patch_size=16,
    max_num_frames=16,
    tubelet_size=2,
    model_name="vit_large",
    crop_size=224,
    pred_depth=6,
    pred_num_heads=None,
    pred_embed_dim=384,
    uniform_power=False,
    use_mask_tokens=True,
    num_mask_tokens=2,
    zero_init_mask_tokens=True,
    use_sdpa=False,
    use_rope=True,
    use_silu=False,
    use_pred_silu=False,
    wide_silu=False,
    use_activation_checkpointing=False,
)


r_path = "/data/wjl/vjepa2/ckpts/vitl.pt"

print(f"Loading checkpoint from {r_path}")
checkpoint = robust_checkpoint_loader(r_path, map_location=torch.device("cpu"))

epoch = checkpoint["epoch"]

# -- loading encoder
pretrained_dict = checkpoint["encoder"]

# 处理键名，移除'module.'前缀
new_pretrained_dict = {}
for k, v in pretrained_dict.items():
    if k.startswith('module.'):
        new_k = k[7:]  # 移除'module.'前缀
        new_pretrained_dict[new_k] = v
    else:
        new_pretrained_dict[k] = v

msg = encoder.load_state_dict(new_pretrained_dict, strict=False)
print(f"loaded pretrained encoder from epoch {epoch} with msg: {msg}")





In [3]:
torch.hub.list('facebookresearch/vjepa2')

Using cache found in /home/wjl/.cache/torch/hub/facebookresearch_vjepa2_main


['vjepa2_ac_vit_giant',
 'vjepa2_preprocessor',
 'vjepa2_vit_giant',
 'vjepa2_vit_giant_384',
 'vjepa2_vit_huge',
 'vjepa2_vit_large']

In [ ]:
# 假设你有本地的 hubconf.py 文件
model = torch.hub.load('/data/wjl/vjepa2/logs/e0.pt', 'vjepa2_vit_large', source='local')



In [26]:

import torch

model = torch.load("/data/wjl/vjepa2/logs/32.8.vitl16-256px-64f_cooldown/latest.pt", map_location="cpu")



In [7]:
import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader
from app.vjepa.utils import init_video_model
import copy

def load_any_pretrained(encoder, pretrained, checkpoint_key="target_encoder"):
    print(f"Loading pretrained model from {pretrained}")
    checkpoint = robust_checkpoint_loader(pretrained, map_location="cpu")
    pretrained_dict = checkpoint[checkpoint_key]
    
    
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    # pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    for k, v in encoder.state_dict().items():
        if k not in pretrained_dict:
            print(f"key '{k}' could not be found in loaded state dict")
        elif pretrained_dict[k].shape != v.shape:
            print(f"{pretrained_dict[k].shape} | {v.shape}")
            print(f"key '{k}' is of different shape in model and loaded state dict")
            exit(1)
            pretrained_dict[k] = v
    msg = encoder.load_state_dict(pretrained_dict, strict=False)
    # print(encoder)
    print(f"loaded pretrained model with msg: {msg}")
    print(f"loaded pretrained encoder from epoch: {checkpoint['epoch']}\n path: {pretrained}")
    del checkpoint
    return encoder

encoder, predictor = init_video_model(
    device="cpu",
    patch_size=16,
    max_num_frames=16,
    tubelet_size=2,
    model_name="vit_huge",
    crop_size=224,
    pred_depth=12,
    pred_num_heads=None,
    pred_embed_dim=384,
    uniform_power=False,
    use_mask_tokens=True,
    num_mask_tokens=10,
    zero_init_mask_tokens=True,
    use_sdpa=False,
    use_rope=True,
    use_silu=False,
    use_pred_silu=False,
    wide_silu=False,
    use_activation_checkpointing=False,
)


target_encoder = copy.deepcopy(encoder)

import torch
from src.utils.checkpoint_loader import robust_checkpoint_loader
from app.vjepa.utils import init_video_model
import copy

pretrained = "/data/wjl/vjepa2/ckpts/vith.pt"
ckpts = robust_checkpoint_loader(pretrained, map_location="cpu")
ckpts["predictor"].keys()



[INFO    ][2025-07-10 13:26:15][root                ][init_video_model         ] MultiSeqWrapper(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed3D(
      (proj): Conv3d(3, 1280, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (blocks): ModuleList(
      (0-31): 32 x Block(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): RoPEAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=5120, out_features=1280, bias=True)
          (drop): Dropout(p=0

odict_keys(['module.backbone.predictor_embed.weight', 'module.backbone.predictor_embed.bias', 'module.backbone.mask_tokens.0', 'module.backbone.mask_tokens.1', 'module.backbone.mask_tokens.2', 'module.backbone.mask_tokens.3', 'module.backbone.mask_tokens.4', 'module.backbone.mask_tokens.5', 'module.backbone.mask_tokens.6', 'module.backbone.mask_tokens.7', 'module.backbone.mask_tokens.8', 'module.backbone.mask_tokens.9', 'module.backbone.predictor_blocks.0.norm1.weight', 'module.backbone.predictor_blocks.0.norm1.bias', 'module.backbone.predictor_blocks.0.attn.qkv.weight', 'module.backbone.predictor_blocks.0.attn.qkv.bias', 'module.backbone.predictor_blocks.0.attn.proj.weight', 'module.backbone.predictor_blocks.0.attn.proj.bias', 'module.backbone.predictor_blocks.0.norm2.weight', 'module.backbone.predictor_blocks.0.norm2.bias', 'module.backbone.predictor_blocks.0.mlp.fc1.weight', 'module.backbone.predictor_blocks.0.mlp.fc1.bias', 'module.backbone.predictor_blocks.0.mlp.fc2.weight', 'modu

In [8]:
encoder = load_any_pretrained(encoder, pretrained, checkpoint_key="encoder")



Loading pretrained model from /data/wjl/vjepa2/ckpts/vith.pt
loaded pretrained model with msg: <All keys matched successfully>
loaded pretrained encoder from epoch: 40
 path: /data/wjl/vjepa2/ckpts/vith.pt


In [9]:
target_encoder = load_any_pretrained(target_encoder, pretrained, checkpoint_key="target_encoder")


Loading pretrained model from /data/wjl/vjepa2/ckpts/vith.pt
loaded pretrained model with msg: <All keys matched successfully>
loaded pretrained encoder from epoch: 40
 path: /data/wjl/vjepa2/ckpts/vith.pt


In [10]:
predictor = load_any_pretrained(predictor, pretrained, checkpoint_key="predictor")


Loading pretrained model from /data/wjl/vjepa2/ckpts/vith.pt
loaded pretrained model with msg: <All keys matched successfully>
loaded pretrained encoder from epoch: 40
 path: /data/wjl/vjepa2/ckpts/vith.pt
